<a href="https://colab.research.google.com/github/guosongnian/knn-life-satisfaction/blob/main/notebooks/05_nearest_neighbors_2_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# k近傍法 (2)
* 今回は、複数の特徴量を使って、k近傍法で予測をおこなう。

## 準備

### インポート

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

### データファイル

* `life_satisfaction_2010_2024.csv` を読み込む。

In [ ]:
from pathlib import Path

data_candidates = [
    Path("data/life_satisfaction_2010_2024.csv"),
    Path("../data/life_satisfaction_2010_2024.csv"),
    Path("life_satisfaction_2010_2024.csv"),
]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find life_satisfaction_2010_2024.csv")

df = pd.read_csv(data_path)
df = df[df['Year'] == 2024]
df = df.set_index('Country')



* 日本をテストデータとして除外し、残りのデータ集合を使う。

In [ ]:
df_train = df.drop(['Japan'])
df_test  = df.loc[['Japan']]

In [ ]:
df_train

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df_train.plot(kind='scatter', x='GDP per capita',      y='Life Satisfaction', ax=ax[0])
df_train.plot(kind='scatter', x='Employment Rate (%)', y='Life Satisfaction', ax=ax[1]);

In [ ]:
X = df_train[['GDP per capita', 'Employment Rate (%)']]
y = df_train['Life Satisfaction']

In [ ]:
X

In [ ]:
y

## 今回の設定: 複数の特徴量を同時に使う
* 前回は、一人当たりのGDPと、雇用率を、別々に使った。
* 今回は、これら二つの特徴量を、同時に使いたい。
  * つまり、(一人当たりのGDP, 雇用率) という2次元ベクトルを使って、生活満足度を予測したい。

### 演習問題1
* 韓国とイタリアの距離を、一人当たりのGDPと雇用率を同時に使って計算したい。
* しかし、下に示す距離の計算方法には、問題がある。どのような問題があるか。

In [ ]:
print(np.linalg.norm(X.loc['South Korea'] - X.loc['Italy']))

#### 解答1
* `GDP per capita` の値は数万ドル規模、`Employment Rate (%)` は 0〜100 の範囲と、**スケール（単位）が大きく異なる**。
* そのため、ユークリッド距離を単純に計算すると、GDPの差が圧倒的に支配してしまい、
  雇用率の情報がほとんど反映されない。
* 以下の例でも分かる通り、距離の大部分がGDPの差（約1800）によるものであり、
  雇用率の差（約7）はほぼ無視されてしまっている。

```
South Korea: GDP=55070, Employment=69.54
Italy:       GDP=53265, Employment=62.20
差:          GDP≈1805,  Employment≈7.3  → 距離≈1805.95
```

* **解決策**: 各特徴量を**標準化**（平均0・標準偏差1に変換）してからユークリッド距離を計算する。

### 演習問題2
* 上で見つけた問題を解決した上で、あらためて、韓国とイタリアの距離を求めてみよう。

In [ ]:
# 標準化: 各特徴量を平均0・標準偏差1に変換する
X_mean = X.mean()
X_std  = X.std()
X_scaled = (X - X_mean) / X_std

print('標準化後の基本統計:')
print(X_scaled.describe().round(4))


In [ ]:
# 韓国とイタリアの距離（標準化後）
dist_scaled = np.linalg.norm(X_scaled.loc['South Korea'] - X_scaled.loc['Italy'])
print(f'標準化前の距離: {np.linalg.norm(X.loc["South Korea"] - X.loc["Italy"]):.2f}')
print(f'標準化後の距離: {dist_scaled:.4f}')
print('→ 両特徴量がほぼ同等に扱われるようになった')

## 訓練データ/検証データ/テストデータ

### テストデータ (test set)
* 最終的にそれについて予測を行なうことで、手法の評価をおこないたいデータを、テストデータと呼ぶ。
* 今回は、日本のデータがテストデータになる。

### 検証データ (validation set)
* 最適な近傍の個数 k を、どうやって求めたらいいだろうか。
* テストデータでの評価は最終評価なので、最後に一度行うだけ（カンニング禁止）。
* そこで、テストデータ以外のデータを使って予測・評価することで k を決める。
* このように **ハイパーパラメータを決めるために使うデータ** を、検証データと呼ぶ。

### 訓練データ (training set)
* 予測を実行するための手掛かりとするデータ集合を、訓練データと呼ぶ。
* k近傍法では、その中から近傍をk個見つけてくるデータ集合が訓練データになる。

## 最適なkの決定
* ここでは、韓国を検証データとして使う。
* 韓国の生活満足度を予測し、最も良い予測値を与える k がいくらか調べる。

### 演習問題3
* 韓国の生活満足度を予測し、最も良い予測値を与えるkを調べよう。

In [ ]:
# ---- k近傍法の手動実装 (sklearnを使わない) ----

def knn_predict(X_train_sc, y_train, x_query_sc, k):
    """標準化済みの訓練データと1件のクエリから予測値を返す。"""
    dists = np.array([
        np.linalg.norm(x_query_sc - X_train_sc.loc[c].values)
        for c in X_train_sc.index
    ])
    nn_indices = np.argsort(dists)[:k]
    return y_train.iloc[nn_indices].mean()


In [ ]:
# 韓国を検証データとして除外した訓練セットで標準化
val_country = 'South Korea'

X_tr_sk  = X.drop(val_country)
y_tr_sk  = y.drop(val_country)
X_val_sk = X.loc[[val_country]]

# 訓練データのみで mean/std を計算（データリーク防止）
mean_sk = X_tr_sk.mean()
std_sk  = X_tr_sk.std()
X_tr_sk_sc  = (X_tr_sk  - mean_sk) / std_sk
X_val_sk_sc = (X_val_sk - mean_sk) / std_sk

# k=1〜len(訓練データ)-1 で予測誤差を計算
k_values = range(1, len(X_tr_sk_sc))
errors_sk = []
for k in k_values:
    pred  = knn_predict(X_tr_sk_sc, y_tr_sk, X_val_sk_sc.values[0], k)
    error = abs(pred - y.loc[val_country])
    errors_sk.append(error)

best_k_sk = list(k_values)[np.argmin(errors_sk)]
print(f'韓国を検証データとした場合の最適k: {best_k_sk}')
print(f'そのときの予測誤差 (MAE): {min(errors_sk):.4f}')

# 可視化
plt.figure(figsize=(7, 4))
plt.plot(list(k_values), errors_sk, marker='o', markersize=4)
plt.axvline(best_k_sk, color='red', linestyle='--', label=f'best k={best_k_sk}')
plt.xlabel('k')
plt.ylabel('予測誤差 (絶対値)')
plt.title('韓国を検証データとした場合のk別予測誤差')
plt.legend()
plt.tight_layout()
plt.show()

## 交差検証 (cross-validation)
* 検証データの取り方を何通りも変えつつ予測手法の評価を繰り返すことで手法の性能を検証することを、交差検証と呼ぶ。

## leave-one-out交差検証 (1)
* 上では、韓国を検証データとして使った。
* 他の国を検証データとしても構わないはずである。
* そこで、他の国の一つ一つを検証データとした場合の、それぞれ最適なkの値を求めてみる。
  * 検証データが1個の場合の交差検証を、**leave-one-out交差検証**と呼ぶ。

### 演習問題4
* 韓国について行ったことと同じことを、他の国についても実行し・・・
* 最適なkの値がどのくらい違ってくるか、調べてみよう。

In [ ]:
# 各国を1つずつ検証データとして、最適なkを求める
best_k_per_country = {}

for val_country in X.index:
    X_tr  = X.drop(val_country)
    y_tr  = y.drop(val_country)
    X_val = X.loc[[val_country]]

    # 訓練データのみで標準化
    mean_c = X_tr.mean()
    std_c  = X_tr.std()
    X_tr_sc  = (X_tr  - mean_c) / std_c
    X_val_sc = (X_val - mean_c) / std_c

    errs = []
    for k in range(1, len(X_tr_sc)):
        pred  = knn_predict(X_tr_sc, y_tr, X_val_sc.values[0], k)
        errs.append(abs(pred - y.loc[val_country]))

    best_k_per_country[val_country] = np.argmin(errs) + 1  # k は1始まり

result_series = pd.Series(best_k_per_country, name='best_k')
print(result_series.to_string())
print(f'\n最頻値: {result_series.mode().values}')
print(f'平均値: {result_series.mean():.1f}')
print(f'中央値: {result_series.median()}')

* 最適なkを、どのようにして決めればいいだろうか？
  * 例えば、各国について得られたkの平均をとることが考えられるが、これは良い方法と言えるだろうか？
  * → 各国での「最適k」はバラつきが大きく、単純な平均は不安定。より良い方法を演習5〜8で考える。

## leave-one-out交差検証 (2)
* 上では国ごとに最適kを求めたが、まとめ方が難しかった。
* より良い方法として、**各kについて全国の予測誤差を平均し、その平均誤差が最小のkを選ぶ**。

### 演習問題5
* 韓国を検証データとする。つまり、韓国について予測を行なう。
* kの値を変えたとき、それぞれ予測誤差がいくらになるか、求めてみよう。

In [ ]:
# 韓国の場合の k 別予測誤差を表示
val_country = 'South Korea'
X_tr  = X.drop(val_country)
y_tr  = y.drop(val_country)
X_val = X.loc[[val_country]]

mean_c = X_tr.mean()
std_c  = X_tr.std()
X_tr_sc  = (X_tr  - mean_c) / std_c
X_val_sc = (X_val - mean_c) / std_c

k_list = list(range(1, len(X_tr_sc)))
err_list = []
for k in k_list:
    pred  = knn_predict(X_tr_sc, y_tr, X_val_sc.values[0], k)
    err_list.append(abs(pred - y.loc[val_country]))

err_sk = pd.Series(err_list, index=k_list, name='South Korea')
print(err_sk.to_frame().T.to_string())

### 演習問題6
* 縦が国名、横がkの値の表を、データフレームとして作った上で・・・
* 日本を除く国の一つ一つを検証データとして評価していくことで・・・
* 各々のkの値で予測誤差がいくらになるかで、表を埋めてみよう。

In [ ]:
# LOO-CV の誤差テーブルを作成
k_list     = list(range(1, len(X)))  # k=1〜(国数-1)
error_table = {}                     # {country: [error_k1, error_k2, ...]}

for val_country in X.index:
    X_tr  = X.drop(val_country)
    y_tr  = y.drop(val_country)
    X_val = X.loc[[val_country]]

    mean_c = X_tr.mean()
    std_c  = X_tr.std()
    X_tr_sc  = (X_tr  - mean_c) / std_c
    X_val_sc = (X_val - mean_c) / std_c

    errs = []
    for k in k_list:
        pred  = knn_predict(X_tr_sc, y_tr, X_val_sc.values[0], k)
        errs.append(abs(pred - y.loc[val_country]))

    error_table[val_country] = errs

# DataFrameに変換: 行=国、列=k
error_df = pd.DataFrame(error_table, index=k_list).T
error_df.columns.name = 'k'
print('誤差テーブル（行=国、列=k）の先頭5列')
error_df.iloc[:, :].round(4)

### 演習問題7
* 上で作った表で、それぞれのkの値について、予測誤差の平均値を求めてみよう。

In [ ]:
# k別の平均予測誤差
mean_errors = error_df.mean(axis=0)  # 行方向（国）で平均
mean_errors.index = mean_errors.index.astype(int)

print('k別の平均予測誤差:')
print(mean_errors.round(4).to_string())

plt.figure(figsize=(8, 4))
mean_errors.plot(marker='o', markersize=4)
plt.xlabel('k')
plt.ylabel('平均予測誤差 (MAE)')
plt.title('LOO-CV: k別の平均予測誤差')
plt.tight_layout()
plt.show()

### 演習問題8
* 上で求めた予測誤差の平均値を、最も小さくするkの値は？

In [ ]:
best_k_loo = int(mean_errors.idxmin())
print(f'LOO-CVで選ばれた最適k: {best_k_loo}')
print(f'そのときの平均予測誤差 (MAE): {mean_errors[best_k_loo]:.4f}')

### 演習問題9
* こうして求めた最適なkを使って、最終的に、日本の生活満足度の予測をしてみよう。
  * ここで初めてテストデータを使う。

In [ ]:
# ここで初めてテストデータ（日本）を使う
# 訓練データ全体（日本を除いた全国）で標準化パラメータを推定
mean_final = X.mean()
std_final  = X.std()
X_train_sc_final = (X - mean_final) / std_final

# テストデータも同じパラメータで標準化
X_test = df_test[['GDP per capita', 'Employment Rate (%)']]
X_test_sc = (X_test - mean_final) / std_final

# 予測
pred_japan = knn_predict(
    X_train_sc_final, y, X_test_sc.values[0], best_k_loo
)
actual_japan = df_test['Life Satisfaction'].values[0]

print(f'使用したk: {best_k_loo}')
print(f'日本の予測生活満足度: {pred_japan:.3f}')
print(f'日本の実際の生活満足度: {actual_japan:.3f}')
print(f'予測誤差 (絶対値): {abs(pred_japan - actual_japan):.3f}')

# k近傍の国を表示
dists = np.array([
    np.linalg.norm(X_test_sc.values[0] - X_train_sc_final.loc[c].values)
    for c in X_train_sc_final.index
])
nn_idx = np.argsort(dists)[:best_k_loo]
print(f'\n日本の{best_k_loo}近傍の国:')
neighbors = pd.DataFrame({
    '距離': dists[nn_idx],
    'Life Satisfaction': y.iloc[nn_idx].values
}, index=y.index[nn_idx])
print(neighbors.round(4))

# 課題
* 上のnotebookを最後まで実践することが、今回の課題です。